# Обучение LoRA для CUCPO Assistant
База: Qwen/Qwen2.5-3B-Instruct
Датасет: qwen_train.jsonl

In [ ]:
# Установка зависимостей
!pip install -q transformers torch peft accelerate datasets sentencepiece

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Загрузка датасета
import json
from datasets import Dataset

with open("qwen_train.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

dataset = Dataset.from_list(data)
print(f"Загружено: {len(dataset)} примеров")

In [ ]:
# Токенизация
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    trust_remote_code=True
)

def preprocess(item):
    # Формируем текст из messages
    text = ""
    for msg in item["messages"]:
        if msg["role"] == "system":
            text = msg["content"] + "\n\n"
        elif msg["role"] == "user":
            text += "USER: " + msg["content"] + "\n"
        elif msg["role"] == "assistant":
            text += "ASSISTANT: " + msg["content"] + "\n"
    return {"text": text}

dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

def tokenize(item):
    return tokenizer(item["text"], truncation=True, max_length=2048)

dataset = dataset.map(tokenize, batched=False)
dataset

In [ ]:
# Настройка LoRA
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Обучение
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./cucpo_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-4,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    warmup_steps=10,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

In [ ]:
# Сохранение модели
model.save_pretrained("./cucpo_lora_output")
tokenizer.save_pretrained("./cucpo_lora_output")

# Скачать
!zip -r cucpo_lora.zip ./cucpo_lora_output
files.download("cucpo_lora.zip")